# Fine-tune Granite 4.0 1B: messy invoices to clean JSON

We teach a small model (**IBM Granite 4.0, 1B parameters**) to read the text of an invoice PDF and return its key fields as **JSON**.
This time we use **Hugging Face**: Transformers, PEFT and TRL, training on the Mac's GPU through PyTorch's **MPS** backend.

In [ ]:
import json
import time

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

from invoice_eval import load_test, messages, parse, score

## 0. Check the machine

Same **MacBook Pro** as lesson 1. PyTorch reaches the Mac's GPU through **MPS** (Metal Performance Shaders): no NVIDIA GPU, no CUDA.

In [ ]:
!system_profiler SPHardwareDataType | grep -E "Model Name|Chip|Memory"

In [ ]:
torch.backends.mps.is_available()

## 1. The test set: 100 invoice PDFs

[`drew-ipp/invoice-extraction-benchmark`](https://huggingface.co/datasets/drew-ipp/invoice-extraction-benchmark) has invoice PDFs with the correct answer for each one.
We use its 100 digital PDFs (with a text layer) from six categories: clean invoices, foreign currencies, tax edge cases, credit notes, multi-page invoices and freight bills. The 13 "hard" PDFs are left out.
`load_test()` in `invoice_eval.py` downloads them and pulls out the text with `pypdf`, the way any PDF tool would.

In [ ]:
test = load_test()
len(test)

In [ ]:
ex = next(e for e in test if e["id"] == "freight-trip-reference-4")
print(ex["text"])

We want **8 fields** back: who sent it, its number and date, the currency as an ISO code, the three amounts, and whether it is an invoice or a credit note.

In [ ]:
ex["json"]

## 2. Make the training data

We never train on the test set. Instead, `make_invoices.py` **writes** messy invoice text the way a PDF comes out (labels and values split apart, page footers, bank details, other dates and reference numbers in the way) together with the JSON it should become.
Because the script writes both, every answer is exactly right. 2,000 invoices, from 11 countries.

In [ ]:
!mkdir -p data && python make_invoices.py 2000 data/train.jsonl 1

In [ ]:
train = [json.loads(line) for line in open("data/train.jsonl")]
print(train[0]["text"])

In [ ]:
train[0]["json"]

## 3. Try the base model

Granite 4.0 1B, straight from IBM. `ask()` sends a batch of invoices at once and decodes greedily, so the answer is the same on every run.

In [ ]:
MODEL = "ibm-granite/granite-4.0-1b"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("mps")

In [ ]:
def ask(model, texts):
    tokenizer.padding_side = "left"
    prompts = [tokenizer.apply_chat_template(messages(t), add_generation_prompt=True, tokenize=False)
               for t in texts]
    batch = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("mps")
    with torch.no_grad():
        out = model.generate(**batch, max_new_tokens=160, do_sample=False)
    return [tokenizer.decode(o[batch.input_ids.shape[1]:], skip_special_tokens=True) for o in out]

In [ ]:
print(ask(model, [ex["text"]])[0])

## 4. Measure it

`score()` checks the 8 fields one by one: amounts to the cent, the date as a real date, the vendor name without "Ltd" or "(Pty)", the invoice number without its label.
An invoice only counts when **all 8** are right, because one wrong field means someone has to fix it by hand.

In [ ]:
from tqdm.auto import tqdm

def evaluate(model, batch=8):
    scores = []
    for k in tqdm(range(0, len(test), batch)):
        part = test[k:k + batch]
        replies = ask(model, [e["text"] for e in part])
        scores += [score(parse(r), e["json"]) for r, e in zip(replies, part)]
    return sum(s == 8 for s in scores), sum(scores)

In [ ]:
base_score = evaluate(model)
base_score

## 5. Fine-tune with LoRA

Each example becomes a **prompt** (the instruction and the invoice text) and a **completion** (the JSON). TRL's `SFTTrainer` then computes the loss on the completion only: the model learns to write the JSON, not to copy the invoice.

In [ ]:
def to_example(row):
    chat = messages(row["text"], row["json"])
    return {"prompt": chat[:1], "completion": chat[1:]}

dataset = Dataset.from_list([to_example(r) for r in train])
dataset[0]["completion"]

LoRA trains small extra matrices on **every linear layer** (`all-linear`) and keeps Granite's own weights frozen.
300 steps of 4 invoices is 1,200 examples. `gradient_checkpointing` recomputes some values instead of storing them, and `bf16` keeps the numbers in 16 bits: both save memory.
`router_aux_loss_coef=0.0`: Granite 4.0's config comes from a family that also has mixture-of-experts models, and TRL would look for experts this 1B model doesn't have.

In [ ]:
lora = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM")
args = SFTConfig(output_dir="checkpoints", max_steps=300, per_device_train_batch_size=4,
                 learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=20,
                 logging_steps=10, save_strategy="no", max_length=2048,
                 gradient_checkpointing=True, bf16=True, router_aux_loss_coef=0.0,
                 dataloader_pin_memory=False, report_to="none")

In [ ]:
trainer = SFTTrainer(model=AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16),
                     args=args, train_dataset=dataset, processing_class=tokenizer, peft_config=lora)
trainer.model.print_trainable_parameters()

In [ ]:
start = time.time()
trainer.train()
train_secs = time.time() - start
gpu_gb = torch.mps.driver_allocated_memory() / 1e9
print(f"{train_secs / 60:.1f} min, GPU memory {gpu_gb:.1f} GB")

In [ ]:
trainer.save_model("adapters")

## 6. Try the tuned model

The same invoice as before. It answers in one line of JSON, so we print it with `indent=2`.

In [ ]:
tuned = trainer.model.eval()
reply = ask(tuned, [ex["text"]])[0]
print(json.dumps(json.loads(reply), indent=2))

## 7. Before vs after

The tuned model on the same 100 test PDFs, then both scores side by side.

In [ ]:
tuned_score = evaluate(tuned)
tuned_score

In [ ]:
from scoreboard import show
show(base_score, tuned_score, train_secs, gpu_gb, args.max_steps, len(train))

## 8. Merge and save

`merge_and_unload()` folds the adapter into the weights: one normal Hugging Face model folder you can load, share or serve.

In [ ]:
merged = tuned.merge_and_unload()
merged.save_pretrained("granite-4.0-1b-invoice")
tokenizer.save_pretrained("granite-4.0-1b-invoice")

## Next: lesson 3

Sort support tickets automatically, with Google's **Gemma 3 270M**: a model about six times smaller than this one.